In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("SalesDataAnalysis") \
    .master("local[*]") \
    .getOrCreate()

print("SparkSession created successfully!")
print("Spark Version:", spark.version)

SparkSession created successfully!
Spark Version: 4.0.5


In [3]:
from google.colab import files

uploaded = files.upload()

Saving raw_sales_source_50_records_for_etl-1.xlsx to raw_sales_source_50_records_for_etl-1.xlsx


In [5]:
import pandas as pd

df = pd.read_excel("raw_sales_source_50_records_for_etl-1.xlsx")

print("Columns:")
print(df.columns.tolist())

print("\nFirst 5 records:")
display(df.head())

Columns:
['order_id', 'order_date', 'customer_id', 'customer_name', 'customer_city', 'customer_state', 'product_id', 'product_name', 'category', 'quantity', 'unit_price', 'discount', 'sales_amount', 'cost_amount', 'profit', 'store_id', 'store_name', 'region', 'payment_method', 'order_status']

First 5 records:


,order_id,order_date,customer_id,customer_name,customer_city,customer_state,product_id,product_name,category,quantity,unit_price,discount,sales_amount,cost_amount,profit,store_id,store_name,region,payment_method,order_status
0,O1001,2026-06-15,C101,Karthik S,Hyderabad,Telangana,P101,Laptop,Electronics,1,55000,0.00,55000,40000,15000,S03,Bengaluru Tech Park,South,Net Banking,Completed
1,O1002,2026-09-17,C102,Vijay Kumar,Chennai,Tamil Nadu,P102,Smartphone,Electronics,4,28000,0.15,95200,84000,11200,S01,Chennai Central,South,Cash,Completed
2,O1003,2026-08-06,C103,Priya Sharma,Delhi,Delhi,P102,Smartphone,Electronics,2,28000,0.00,56000,42000,14000,S05,Mumbai Central,West,Net Banking,Pending
3,O1004,2026-01-26,C104,Anitha R,Chennai,Tamil Nadu,P109,Backpack,Accessories,2,2200,0.10,3960,2600,1360,S04,Hyderabad Plaza,South,Cash,Cancelled
4,O1005,2026-03-02,C105,Priya Sharma,Bangalore,Karnataka,P109,Backpack,Accessories,2,2200,0.00,4400,2600,1800,S05,Mumbai Central,West,Net Banking,Completed


In [6]:
sales_csv = df.rename(columns={
    "order_id": "transaction_id",
    "product_name": "product",
    "sales_amount": "amount",
    "customer_city": "region"
})

sales_csv = sales_csv[
    ["transaction_id", "customer_id", "product", "amount", "region"]
]

sales_csv.to_csv("sales_data.csv", index=False)

print("CSV created successfully!")
display(sales_csv.head())

CSV created successfully!


,transaction_id,customer_id,product,amount,region,region
0,O1001,C101,Laptop,55000,Hyderabad,South
1,O1002,C102,Smartphone,95200,Chennai,South
2,O1003,C103,Smartphone,56000,Delhi,West
3,O1004,C104,Backpack,3960,Chennai,South
4,O1005,C105,Backpack,4400,Bangalore,West


In [7]:
from google.colab import files
files.download("sales_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
sales_df = spark.read.csv(
    "sales_data.csv",
    header=True,
    inferSchema=True
)

print("CSV loaded successfully!")

sales_df.printSchema()
sales_df.show(10)

CSV loaded successfully!
root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- region4: string (nullable = true)
 |-- region5: string (nullable = true)

+--------------+-----------+----------+------+---------+-------+
|transaction_id|customer_id|   product|amount|  region4|region5|
+--------------+-----------+----------+------+---------+-------+
|         O1001|       C101|    Laptop| 55000|Hyderabad|  South|
|         O1002|       C102|Smartphone| 95200|  Chennai|  South|
|         O1003|       C103|Smartphone| 56000|    Delhi|   West|
|         O1004|       C104|  Backpack|  3960|  Chennai|  South|
|         O1005|       C105|  Backpack|  4400|Bangalore|   West|
|         O1006|       C106|Smartphone|140000|     Pune|   West|
|         O1007|       C107|    Tablet|115200|   Mumbai|  South|
|         O1008|       C108|   Monitor| 63750|    Delhi|  South|
|        

In [9]:
filtered_df = sales_df.filter(sales_df.amount > 0)

print("Filtered records:")
filtered_df.show()

Filtered records:
+--------------+-----------+----------+------+----------+-------+
|transaction_id|customer_id|   product|amount|   region4|region5|
+--------------+-----------+----------+------+----------+-------+
|         O1001|       C101|    Laptop| 55000| Hyderabad|  South|
|         O1002|       C102|Smartphone| 95200|   Chennai|  South|
|         O1003|       C103|Smartphone| 56000|     Delhi|   West|
|         O1004|       C104|  Backpack|  3960|   Chennai|  South|
|         O1005|       C105|  Backpack|  4400| Bangalore|   West|
|         O1006|       C106|Smartphone|140000|      Pune|   West|
|         O1007|       C107|    Tablet|115200|    Mumbai|  South|
|         O1008|       C108|   Monitor| 63750|     Delhi|  South|
|         O1009|       C109|      Desk| 21600|      Pune|  South|
|         O1010|       C110|    Tablet| 86400| Bangalore|   West|
|         O1011|       C111|Smartphone| 25200|    Mumbai|  South|
|         O1012|       C112|   Monitor| 54000|    Mumbai| 

In [10]:
total_transactions = filtered_df.count()

print("Total valid transactions:", total_transactions)

Total valid transactions: 50


In [12]:
print(sales_df.columns)

['transaction_id', 'customer_id', 'product', 'amount', 'region4', 'region5']


In [13]:
sales_df = sales_df.withColumnRenamed("region4", "region")

# Keep only the five required columns
sales_df = sales_df.select(
    "transaction_id",
    "customer_id",
    "product",
    "amount",
    "region"
)

sales_df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- region: string (nullable = true)



In [14]:
from pyspark.sql.functions import sum, count, avg

filtered_df = sales_df.filter(sales_df.amount > 0)

region_sales = filtered_df.groupBy("region").agg(
    sum("amount").alias("total_sales"),
    count("transaction_id").alias("total_transactions"),
    avg("amount").alias("average_sales")
)

region_sales.show()

+----------+-----------+------------------+------------------+
|    region|total_sales|total_transactions|     average_sales|
+----------+-----------+------------------+------------------+
| Bangalore|     358925|                 7|           51275.0|
|   Chennai|     213960|                 7|30565.714285714286|
|    Mumbai|     249030|                 7| 35575.71428571428|
|  Chennai |      21675|                 1|           21675.0|
|   Madurai|      96850|                 4|           24212.5|
|      Pune|     340050|                 5|           68010.0|
|     Delhi|     298710|                 5|           59742.0|
|Coimbatore|     137590|                 6|22931.666666666668|
| Bengaluru|     128900|                 3|42966.666666666664|
| Hyderabad|     168435|                 5|           33687.0|
+----------+-----------+------------------+------------------+



In [15]:
# Write aggregated results to Parquet
region_sales.write.mode("overwrite").parquet("sales_by_region.parquet")

print("Parquet file created successfully!")

Parquet file created successfully!


In [16]:
parquet_df = spark.read.parquet("sales_by_region.parquet")

print("Parquet data verification:")
parquet_df.show()

Parquet data verification:
+----------+-----------+------------------+------------------+
|    region|total_sales|total_transactions|     average_sales|
+----------+-----------+------------------+------------------+
| Bangalore|     358925|                 7|           51275.0|
|   Chennai|     213960|                 7|30565.714285714286|
|    Mumbai|     249030|                 7| 35575.71428571428|
|  Chennai |      21675|                 1|           21675.0|
|   Madurai|      96850|                 4|           24212.5|
|      Pune|     340050|                 5|           68010.0|
|     Delhi|     298710|                 5|           59742.0|
|Coimbatore|     137590|                 6|22931.666666666668|
| Bengaluru|     128900|                 3|42966.666666666664|
| Hyderabad|     168435|                 5|           33687.0|
+----------+-----------+------------------+------------------+



In [17]:
import shutil

shutil.make_archive(
    "sales_by_region",
    "zip",
    "sales_by_region.parquet"
)

from google.colab import files
files.download("sales_by_region.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>